In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from glob import glob
from os.path import join as opj
import altair as alt
import re
from functools import partial
import yaml

In [ ]:
def get_metric(path:str, keys:tuple):
    metrics = pd.read_csv(path)
    return metrics.loc[keys]

In [ ]:
exp_root1 = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/scsrh_ibot*/"
#exp1 = glob(opj(exp_root1, "*/evals/*SE_epoch*/results/metrics.csv"))
exp_root2 = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/hibot_tile*/"
#exp2 = glob(opj(exp_root2, "*/evals/*SE_epoch*/results/metrics.csv"))

exp_root3 = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/"
exp3 = glob(opj(exp_root3, "*/models/eval/training_*/*smpt_SE_epoch*/results/metrics.csv"))
exp_root4 = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/"
exp4 = glob(opj(exp_root3, "*/models/eval/training_*/*smcr24_SE_epoch*/results/metrics.csv"))


metrics = pd.DataFrame(exp3+exp4, columns=["path"], dtype=str)

exp_name_map = {
    "a9ca1d0f": "dinov2_tile_nouint8_lr5e4_bs704x2",
    "2e7833f0": "dinov2_tile_nouint8_lr1e3_bs704x2",
    "e5b62e88": "dinov2_tile_nouint8_lr2e3_bs704x2",
    "0dad5a88": "dinov2_tile_nouint8_lr5e4_nouint8_bs704x2",
    "ee7090f7": "dinov2_tile_nouint8_lr1e3_p65536_bs384x2",
    

    "789ec8bc": "dinov2_tile_touint8_lr2e4_bs704x2",
    "dc02b1a3": "dinov2_tile_nouint8_lr2e4_bs704x2",
    
    "07b593f1": "dinov2_tile_ctxor_lx1_lr2e4_nouint8_bs704x2",
    "36a026bb": "dinov2_tile_ctxor_lx2_lr4e4_nouint8_bs704x2",
    "fc368dd8": "dinov2_tile_ctxor_lx4_lr4e4_nouint8_bs704x2",
    "643515de": "dinov2_tile_ctxor_lx8_lr4e4_nouint8_bs704x2",
    "94ba456f": "dinov2_tile_ctxor_lx8_lr8e4_nouint8_bs704x2",
    "8295200e": "dinov2_tile_ctxor_lx16_lr4e4_nouint8_bs704x2",
    "bf9459ca": "dinov2_tile_ctxor_lx16_lr8e4_nouint8_bs704x2",
    "f327d9cf": "dinov2_tile_ctxor_lx24_lr8e4_nouint8_bs704x2",
    "30cd9f78": "dinov2_tile_ctxor_lx16_lr2e3_nouint8_bs704x2",
    "28c906fb": "dinov2_tile_ctxor_lx16_lr4e3_nouint8_bs704x2",
    "fb392e35": "dinov2_tile_ctxor_lx32_lr8e4_nouint8_bs704x2",
    "d6a5cc8d": "dinov2_tile_ctxor_lx8_lr2e3_nouint8_bs704x2",
    "dc0bfcd4": "dinov2_tile_ctxor_lx16_lr2e3_bottleneck384_nouint8_bs704x2",
    "8ac664bf": "dinov2_tile_ctxor_lx128_lr1e3_bottleneck384_nouint8_bs704x2",
    "eb8ee70b": "dinov2_tile_ctxor_lx64_lr2e3_bottleneck384_nouint8_bs704x2",
    "fc70ee5d": "dinov2_tile_ctxor_lx32_lr2e3_bottleneck384_nouint8_bs704x2",
    "9baaa202": "dinov2_tile_ctxor_lx64_lr1e3_bottleneck384_nouint8_bs704x2",
    '6f51b8b5': "dinov2_tile_ctxor_lx512_lr1e3_bottleneck384_nouint8_bs704x2",
    '91b0acd7': "dinov2_tile_ctxor_lx256_lr1e3_bottleneck384_nouint8_bs704x2",
    '216430e6': "ctx_only_lx128_lr1e3_bottleneck384_nouint8_bs704x2",
    
    "f07c61f0": "dinov2_layernorm_aggressive_lr5e4",
    "b644fb81": "dinov2_nolayernorm_aggressive_lr5e4",
}


ignore_exp_name_map = { # generally not useful, early exps
    "fdb5c85b": "ibot_both_MC_lr0.0005",
    "5b0f78d2": "ibot_both_MC_lr0.001",
    "ad3cd942": "ibot_tile_MC_lr0.0005",
    #"b050cd38": "ibot_tile_MC_lr0.001",
    #"c647ab60": "ibot_tile_1+1_lr0.001",
    #"e1eb758d": "ibot_tile_1+1_lr0.0005",
    "c5418ada": "ibot_patch_MC_lr0.0005",
    "e61bc551": "ibot_patch_MC_lr0.001",
    "0c414c7f": "ibot_tile1_patch0.5_lr0.001",
    "0b8bf05d": "ibot_tile1_patch0.5_lr0.0005",
    "e8e349f7": "cell_centric_rmbg",
    "b9094eab": "cell_centric_smpt_lr0.0005_bak",
    "ec1b7158": "fix_aug_lr0.0005",
    "3e51ec0d": "hibot_patch_1_0.5_lr0.0005",
    "c413ae5e": "cell_centric_smpt_lr0.0005",
    "d674da07": "dinov2_tile_touint8_bs704x2_longer", # killed early for other exp
}


ignore_exp_name_map.update({ 
    "3ecf68cd": "dinov2_tile_nouint8_lr2e4_bs704x2_longer",
    "b3cb25a0": "dinov2_tile_touint8_lr2e4_bs704x4_longer",
}) 

ignore_exp_name_map.update({  #natural image pretrained
    "c118fd3f": "dinov2_tile_pretrained_bs640x2",
    "59d9ab97": "dinov2_tile_pretrained_0.2_bs640x2",
    "94985a33": "dinov2_tile_pretrained_flatlr_bs640x2",
    "075589c9": "dinov2_tile_pretrained_new_bs640x2",
})  

ignore_exp_name_map.update({ # ibot baselines
    "b050cd38": "ibot_initial_tile_reshaped_MC_lr0.001",
    "99ad27e1": "cell_centric_smpt_lr0.001",
    "e8f6b9a0": "fix_aug_lr0.001",
})   

ignore_exp_name_map.update({ # dinov2 small BS baselines
    "8048d9e9": "dinov2_tile_touint8_bs288x2",
    "7bb4ed40": "dinov2_tile_nouint8_bs288x2",
})

ignore_exp_name_map.update({ # ibot tile-patch ratio tuning
    "ca769ce0": "hibot_patch_1_0.5_lr0.001",
    "ad786077": "hibot_patch_82_lr0.001",
    "8a37b2a3": "hibot_patch_64_lr0.001",
    "00fc7bfe": "hibot_patch_73_lr0.001",
    "44614549": "hibot_patch_91_lr0.001",
    "a214019f": "hibot_patch_100_lr0.001",
})

exp_name_map =  pd.DataFrame.from_dict(exp_name_map,orient='index',columns=["desc"])
exp_name_map.index.name="exp"
exp_name_map = exp_name_map.reset_index()

In [ ]:
metrics["exp"] = metrics["path"].str.replace("^"+exp_root1.replace("*", "[a-z_]*"), "", regex=True).str.replace("^"+exp_root2.replace("*", "[a-z_]*"), "", regex=True).str.replace("^"+exp_root3.replace("*", "[a-z_]*"), "", regex=True).str.split("_",expand=True)[0]
metrics["epoch"] = metrics["path"].str.extract(r"epoch(\d+)",expand=False).astype(int)
metrics["iter"] = metrics["path"].str.extract(r"step(\d+)",expand=False).astype(int)
metrics["eval_method"] = metrics["path"].str.extract(r"(smpt|smcr24)",expand=False)
metrics["cell_mca"] =  metrics["path"].apply(partial(get_metric, keys=(0, "mca")))
metrics["slide_mca"] =  metrics["path"].apply(partial(get_metric, keys=(1, "mca")))

metrics=metrics.merge(exp_name_map, on="exp", how="left")
metrics["exp_eval"] = metrics["desc"]#.str.cat(metrics["eval_method"], sep="_")
metrics["exp_eval"] = metrics["exp_eval"].str.removeprefix("dinov2_tile_")

In [ ]:
set(metrics["exp"].drop_duplicates().tolist()).difference(exp_name_map["exp"].tolist()).difference(ignore_exp_name_map.keys())

In [ ]:
metrics_plot = metrics[~metrics["exp"].isin(ignore_exp_name_map.keys())]
#metrics_plot = metrics

In [ ]:
class_selection = alt.selection_point(fields=["exp_eval"], bind='legend')
op = alt.condition( class_selection, alt.value(1.0), alt.value(0.1))

chart_smpt = alt.Chart(metrics_plot).mark_line(point=True).encode(
    x=alt.X('iter:Q', axis=alt.Axis(tickSize=0), title='Iteration'),
    y=alt.Y("cell_mca:Q", axis=alt.Axis(tickSize=0), title='Cell MCA'),
    color="exp_eval:N", # Color encodes different functions
    tooltip=["epoch", "iter", "exp_eval", "cell_mca"],
    opacity=op,
    column="eval_method:N"
).properties(
    width=300,
    height=600,
).add_params(
    class_selection
).configure_axis(
    labelFontSize=24,
    titleFontSize=24
).interactive()
chart_smpt

In [ ]:
chart_smpt.save('metrics_2505a.html')

In [ ]:
class_selection = alt.selection_point(fields=["exp_eval"], bind='legend')
op = alt.condition( class_selection, alt.value(1.0), alt.value(0.1))

chart_smpt = alt.Chart(metrics_plot[metrics_plot["eval_method"]=="smpt"]).mark_line(point=True).encode(
    x=alt.X('iter:Q', axis=alt.Axis(tickSize=0), title='Iteration'),
    y=alt.Y("cell_mca:Q", axis=alt.Axis(tickSize=0), title='Cell MCA'),
    color="exp_eval:N", # Color encodes different functions
    tooltip=["epoch", "iter", "exp_eval", "cell_mca"],
    opacity=op,
)

chart_smcr = alt.Chart(metrics_plot[metrics_plot["eval_method"]=="smcr24"]).mark_line(point=True).encode(
    x=alt.X('iter:Q', axis=alt.Axis(tickSize=0), title='Iteration'),
    y=alt.Y("cell_mca:Q", axis=alt.Axis(tickSize=0), title='Cell MCA'),
    color="exp_eval:N", # Color encodes different functions
    tooltip=["epoch", "iter", "exp_eval", "cell_mca"],
    opacity=op,
    shape=alt.value('triangle-up')
)

chart_baselines = alt.Chart(
    pd.DataFrame({
        #'cell_mca': [0.331143, 0.448209, 0.3503, 0.5213,.3466,.5978,.3436,.6095],
        #"exp_eval": ["UNI_rmbg", "UNI_smpt", "IBOT_rmbg", "IBOT_smpt","HIBOT_rmbg", "HIBOT_smpt","CC_rmbg", "CC_smpt"]})
        'cell_mca': [0.448209,  0.5213,.5978,.6095],
        "exp_eval": ["UNI_smpt", "IBOT_smpt", "HIBOT_smpt", "CC_smpt"]})

).mark_rule().encode(
    y='cell_mca:Q',
    color='exp_eval:N',
    tooltip=["exp_eval", "cell_mca"],
    opacity=op,
)


In [ ]:
(chart_smpt + chart_smcr + chart_baselines).properties(
    width=700,
    height=700,
    #title='Cell MCA'
).add_params(
    class_selection
).configure_axis(
    labelFontSize=24,
    titleFontSize=24
).interactive(bind_y=True, bind_x=True)#.save('metrics_2504c.html')

In [ ]:
class_selection = alt.selection_point(fields=["exp_eval"], bind='legend')
op = alt.condition( class_selection, alt.value(1.0), alt.value(0.1))

chart_smpt = alt.Chart(metrics_plot[metrics_plot["eval_method"]=="smpt"]).mark_line(point=True).encode(
    x=alt.X('iter:Q', axis=alt.Axis(tickSize=0), title='Iteration'),
    y=alt.Y("slide_mca:Q", axis=alt.Axis(tickSize=0), title='Cell MCA'),
    color="exp_eval:N", # Color encodes different functions
    tooltip=["epoch", "iter", "exp_eval", "cell_mca"],
    opacity=op,
)

chart_smcr = alt.Chart(metrics_plot[metrics_plot["eval_method"]=="smcr24"]).mark_line(point=True).encode(
    x=alt.X('iter:Q', axis=alt.Axis(tickSize=0), title='Iteration'),
    y=alt.Y("slide_mca:Q", axis=alt.Axis(tickSize=0), title='Cell MCA'),
    color="exp_eval:N", # Color encodes different functions
    tooltip=["epoch", "iter", "exp_eval", "cell_mca"],
    opacity=op,
    shape=alt.value('triangle-up')
)
chart_baselines = alt.Chart(
    pd.DataFrame({'slide_mca': [0.431195, 0.53458], "exp_eval": ["uni_rmbg", "uni_smpt"]})
).mark_rule().encode(
    y='slide_mca',
    color='exp_eval:N',
    tooltip=["exp_eval", "slide_mca"],
)


In [ ]:
(chart_smpt + chart_smcr + chart_baselines).properties(
    width=700,
    height=700,
    title='Slide MCA'
).add_params(
    class_selection
).configure_axis(
    labelFontSize=16,
    titleFontSize=16
)

# Get to evals

In [ ]:
htp_key = "data/transform/test/params/base_aug_params/mask_params/how_to_process"

exp_root3 = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/"
exp3 = glob(opj(exp_root3, "*/models/eval/training_*/teacher_checkpoint.pth"))
avail_ckpts = pd.DataFrame(exp3, columns=["path"])

def get_eval_key(x:str):
    if re.findall(r"training_[\d]+",x):
        return (
        re.sub("^" + exp_root3.replace("*", "[a-z_]*"), "", x).split("_")[0], re.findall(r"training_[\d]+",x)[0].replace("training_", "epoch0-step"))
    else:
        return (None, None)
def get_comment(x, do_smcr=False):
    if do_smcr:
        htp = "smcr24"
    else:
        htp="smpt"
        #if x[htp_key] == "small_patch":
        #    htp = "smpt"
        #else:
        #    htp = "rmbg"

        
    return f"{htp}_SE_{x['eval_key'][1]}"

avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains(r"training_[\d]+9999")]
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("789ec8bc")]
avail_ckpts = avail_ckpts[~avail_ckpts["path"].str.contains("40162645")]
avail_ckpts = avail_ckpts[~avail_ckpts["path"].str.contains("326cf121")]
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("(59d9ab97|94985a33|075589c9)")]
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("(3ecf68cd|dc02b1a3)")] #no uint8
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("b3cb25a0")] # to uint8 
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("53efe3f4|fdfc0951|9f10086c|a16daabf|11d355ff|1ac3699d")] # mcmcm
#avail_ckpts =  avail_ckpts[avail_ckpts["path"].str.contains("07b593f1|36a026bb|fc368dd8|643515de|8295200e|94ba456f|bf9459ca")] 
#avail_ckpts =  avail_ckpts[avail_ckpts["path"].str.contains("3ecf68cd|dc02b1a3|07b593f1|36a026bb|fc368dd8|643515de|94ba456f|8295200e|bf9459ca|f327d9cf|30cd9f78|28c906fb|2e7833f0|a9ca1d0f|0dad5a88|fb392e35|d6a5cc8d|e5b62e88|dc0bfcd4|ee7090f7|fc70ee5d|eb8ee70b|9baaa202|8ac664bf|91b0acd7|216430e6|6f51b8b5")] 
#avail_ckpts =  avail_ckpts[avail_ckpts["path"].str.contains("ee7090f7")] 
avail_ckpts =  avail_ckpts[avail_ckpts["path"].str.contains("c39ba723|dc758666|be570ae0")] 

avail_ckpts["eval_key"] = avail_ckpts["path"].apply(get_eval_key)
already_evaled = set(metrics["path"].apply(get_eval_key).to_list())
to_eval = avail_ckpts[~avail_ckpts["eval_key"].isin(already_evaled)]


to_eval["lightning_module/params/pretrained_weights"] = to_eval["path"]#.str.replace("^"+exp_root3.replace("*", "[a-z_]*"), "", regex=True)
to_eval["infra/exp_root"] = to_eval["path"].str.split("/models/", expand=True)[0].str.split("/", expand=True).iloc[:,6:-1].agg('/'.join, axis=1)
to_eval[htp_key] = [["small_patch"]]*len(to_eval)


to_eval1 = to_eval.explode(htp_key)
to_eval1["infra/comment"] = to_eval.apply(partial(get_comment, do_smcr=True), axis=1)
to_eval1 = to_eval1.drop(["eval_key", "path"], axis=1)


to_eval2 = to_eval.explode(htp_key)
to_eval2["infra/comment"] = to_eval.apply(partial(get_comment, do_smcr=False), axis=1)
to_eval2 = to_eval2.drop(["eval_key", "path"], axis=1)

print(len(to_eval))
print(yaml.dump(to_eval1.to_dict(orient="list")))
print(yaml.dump(to_eval2.to_dict(orient="list")))

In [ ]:
htp_key = "data/transform/test/params/base_aug_params/mask_params/how_to_process"
def get_eval_key(x:str):
    return (
        re.sub("^" + exp_root.replace("*", "[a-z_]*"), "", x).split("_")[0], re.findall(r"epoch[\d]+-step[\d]+",x)[0])
def get_comment(x):
    if x[htp_key] == "small_patch":
        htp = "smpt"
    else:
        htp = "rmbg"
        
    return f"{htp}_SE_{x['eval_key'][1]}"

In [ ]:
exp_root = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/hibot_tile/"

avail_ckpts = pd.DataFrame(glob(opj(exp_root, r"*/models/*.ckpt")), columns=["path"])
# =====
#avail_ckpts = avail_ckpts[~avail_ckpts["path"].str.contains(r"c647ab60|e1eb758d")]
#avail_ckpts = avail_ckpts[~avail_ckpts["path"].str.contains("scsrh_ibot_cell_centric")]
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains(r"epoch[\d]*[4|9]-")]
# -----
#avail_ckpts = avail_ckpts[avail_ckpts["path"].str.contains("scsrh_ibot_cell_centric")]
# =====

avail_ckpts["eval_key"] = avail_ckpts["path"].apply(get_eval_key)
already_evaled = set(metrics["path"].apply(get_eval_key).to_list())
to_eval = avail_ckpts[~avail_ckpts["eval_key"].isin(already_evaled)]

if not len(to_eval): print("NONE")

In [ ]:
to_eval["testing/ckpt_path"] = to_eval["path"].str.replace("^"+exp_root.replace("*", "[a-z_]*"), "", regex=True)
to_eval["infra/exp_root"] = to_eval["path"].str.split("/models/", expand=True)[0].str.split("/", expand=True).iloc[:,6:-1].agg('/'.join, axis=1)
to_eval[htp_key] = [["small_patch", "remove_background"]]*len(to_eval)


to_eval = to_eval.explode(htp_key)
to_eval["infra/comment"] = to_eval.apply(get_comment, axis=1)

to_eval = to_eval.drop(["eval_key", "path"], axis=1)

print(len(to_eval))
print(yaml.dump(to_eval.to_dict(orient="list")))

In [ ]:
len(ckpt_cand)

In [ ]:
len(avail_ckpts)

In [ ]:
from PIL import Image
import io
import base64

In [ ]:
def im_to_bytestr(image):
    output = io.BytesIO()
    image.save(output, format='JPEG')
    return "data:image/jpeg;base64," + base64.b64encode(
        output.getvalue()).decode()

def get_concat_v(im1, im2):
    dst = Image.new('RGB', (im1.width + im2.width, im1.height))
    dst.paste(im1, (0, 0))
    dst.paste(im2, (im1.width, 0))
    return dst

def get_tsue(x):
    tsne_path = x["path"].replace("metrics.csv", "interactive_tsne.png")
    return im_to_bytestr(Image.open(tsne_path).convert("RGB").resize((256,256),0))    
    
def get_im_thumb_str(x):
    thumb_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/train/sep_viz"
    #print(opj(thumb_root, f"{x['method']}", f"t{x['perturb_width']}*.pt.png"))
    train_thumb_im = glob(opj(thumb_root, f"{x['method']}", f"t{x['perturb_width']}*.pt.png"))
    assert len(train_thumb_im) == 1
    
    thumb_im = glob(opj(thumb_root, f"{x['method']}", f"v{x['perturb_width']}*.pt.png"))
    assert len(thumb_im) == 1
    return im_to_bytestr(get_concat_v(Image.open(train_thumb_im[0]), Image.open(thumb_im[0])))

In [ ]:
def get_one_set_exp(eval_set_root, eval_set_pattern, width_pattern, method_name, get_im=False):
    print(opj(eval_set_root, eval_set_pattern, "results", "metrics.csv"))
    evals = pd.DataFrame(glob(opj(eval_set_root, eval_set_pattern, "results", "metrics.csv")), columns=["path"])
    evals["perturb_width"] = evals["path"].apply(lambda x: int(re.findall(width_pattern, x)[0]))
    evals["cell_mca"] = evals["path"].apply(partial(get_metric, keys=(0, "mca")))
    evals["method"] = method_name
    if get_im:
        evals["thumb"] = evals.apply(get_im_thumb_str, axis=1)
        evals["image"] = evals.apply(get_tsue, axis=1)
    return evals.sort_values("perturb_width").drop("path", axis=1)

In [ ]:
ckpt_root = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/643515de_Apr24-13-21-52_sd1000_dev_tune0/models/eval/training_39999"
sep_token_tv = get_one_set_exp(opj(ckpt_root, "sep_token_tv"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_T(\d).*", method_name="sep_token_tv", get_im=True)
sep_token_v = get_one_set_exp(opj(ckpt_root, "sep_token_v"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_T(\d).*", method_name="sep_token_v", get_im=True)
sep_mean_tv = get_one_set_exp(opj(ckpt_root, "sep_mean_tv"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_M(\d).*", method_name="sep_mean_tv", get_im=True)
sep_mean_v = get_one_set_exp(opj(ckpt_root, "sep_mean_v"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_T(\d).*", method_name="sep_mean_v", get_im=True)
sep_resize_tv = get_one_set_exp(opj(ckpt_root, "sep_resize_tv"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_R(\d).*", method_name="sep_resize_tv", get_im=True)
sep_resize_v = get_one_set_exp(opj(ckpt_root, "sep_resize_v"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_R(\d).*", method_name="sep_resize_v", get_im=True)

In [ ]:
all_metrics = pd.concat([sep_token_tv, sep_token_v,sep_mean_tv,sep_mean_v, sep_resize_tv, sep_resize_v])
all_metrics["method"] = all_metrics["method"].str.removeprefix("sep_")
all_metrics["method"] = "PFM_"+all_metrics["method"]


In [ ]:
class_selection = alt.selection_point(fields=["method"], bind='legend')
op = alt.condition(class_selection, alt.value(1.0), alt.value(0.05))

chart = alt.Chart(all_metrics[::-1]).mark_line(point=True).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    tooltip=["image", "method","cell_mca"],
    opacity=op,
).add_params(
    class_selection
).interactive()

chart_compare = alt.Chart(all_metrics[::-1].drop("image", axis=1).rename({"thumb":"image"}, axis=1)).mark_line(point=True).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    tooltip=["image", "method","cell_mca"],
    opacity=op,
).add_params(
    class_selection
).interactive()

size_slider = alt.binding_range(min=4, max=128, step=4, name="Image Size:")
size_param = alt.param(name="im_size", bind=size_slider, value=64)

images = alt.Chart(all_metrics[::-1]).mark_image(height=alt.expr("im_size"), width=alt.expr("im_size")).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    url="thumb",
    tooltip=["image", "method", "cell_mca"],
    opacity=op   
).add_params(
    class_selection,
    size_param
).interactive()

(chart + images).properties(
    width=700,
    height=700,
    title='Cell MCA'
).configure_axis(
    labelFontSize=16,
    titleFontSize=16
)#.save('cell_mca_perturb_eval.html')

In [ ]:
ckpt_root = "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/3ecf68cd_Apr13-02-20-38_sd1000_dev_tune0/models/eval/training_624999"
sep_token_tv = get_one_set_exp(opj(ckpt_root, "sep_token_tv"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_T(\d).*", method_name="sep_token_tv", get_im=True)
sep_token_v = get_one_set_exp(opj(ckpt_root, "sep_token_v"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_T(\d).*", method_name="sep_token_v", get_im=True)
sep_resize_tv = get_one_set_exp(opj(ckpt_root, "sep_resize_tv"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_R(\d).*", method_name="sep_resize_tv", get_im=True)
sep_resize_v = get_one_set_exp(opj(ckpt_root, "sep_resize_v"),
                eval_set_pattern="*_SEP*", width_pattern=".*SEP_R(\d).*", method_name="sep_resize_v", get_im=True)

In [ ]:
all_metrics = pd.concat([sep_token_tv, sep_token_v, sep_resize_tv, sep_resize_v])
all_metrics["method"] = all_metrics["method"].str.removeprefix("sep_")
all_metrics["method"] = "DINOv2_"+all_metrics["method"]

In [ ]:
class_selection = alt.selection_point(fields=["method"], bind='legend')
op = alt.condition(class_selection, alt.value(1.0), alt.value(0.05))

chart1 = alt.Chart(all_metrics[::-1]).mark_line(point=True).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    tooltip=["image", "method","cell_mca"],
    opacity=op,
).add_params(
    class_selection
).interactive()

chart1_compare = alt.Chart(all_metrics[::-1].drop("image", axis=1).rename({"thumb":"image"}, axis=1)).mark_line(point=True).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    tooltip=["image", "method","cell_mca"],
    opacity=op,
).add_params(
    class_selection
).interactive()

size_slider = alt.binding_range(min=4, max=128, step=4, name="Image Size:")
size_param = alt.param(name="im_size", bind=size_slider, value=64)

images1 = alt.Chart(all_metrics[::-1]).mark_image(height=alt.expr("im_size"), width=alt.expr("im_size")).encode(
    x=alt.X('perturb_width', axis=alt.Axis(tickSize=0)),
    y=alt.Y("cell_mca", axis=alt.Axis(tickSize=0)),
    color="method:N", # Color encodes different functions
    url="thumb",
    tooltip=["image", "method", "cell_mca"],
    opacity=op   
).add_params(
    class_selection,
    size_param
).interactive()

(chart1 + images1).properties(
    width=700,
    height=700,
    title='Cell MCA'
).configure_axis(
    labelFontSize=16,
    titleFontSize=16
)#.save('NO_PATCHFM_cell_mca_perturb_eval.html')

In [ ]:
(chart_compare + chart1_compare).properties(
    width=700,
    height=700,
    title='Cell MCA'
).configure_axis(
    labelFontSize=16,
    titleFontSize=16
)#.save('COMPARE_cell_mca_perturb_eval.html')